In [1]:
# Imports and config

import os, json, torch, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision.transforms.functional as F
from torch.cuda.amp import autocast, GradScaler

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f"Device     : {DEVICE}")
print(f"GPUs       : {NUM_GPUS}")
print(f"PyTorch    : {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print("Mode       : FROM SCRATCH (no pretrained weights)")

# Top-5 category mapping (1-indexed, 0 = background)
TOP5_CATEGORIES = {1: 1, 8: 2, 7: 3, 2: 4, 9: 5}
IDX_TO_NAME     = {1: 'short_sleeve_top', 2: 'trousers', 3: 'shorts',
                   4: 'long_sleeve_top',  5: 'skirt'}
NUM_CLASSES     = 6  # 5 classes + 1 background

DATA_ROOT = '/kaggle/input/datasets/varun000reddy/'
TRAIN_IMG = DATA_ROOT + 'training/train/image/'
TRAIN_ANN = DATA_ROOT + 'training/train/annos/'
VAL_IMG   = DATA_ROOT + 'validation/validation/image/'
VAL_ANN   = DATA_ROOT + 'validation/validation/annos/'
SAVE_DIR  = '/kaggle/working/checkpoints_scratch/'
os.makedirs(SAVE_DIR, exist_ok=True)

Device     : cuda
GPUs       : 2
PyTorch    : 2.10.0+cu128
Torchvision: 0.25.0+cu128
Mode       : FROM SCRATCH (no pretrained weights)


In [2]:
# Dataset Class

from torch.utils.data import Dataset, DataLoader
import skimage.draw

class ApparelMaskRCNNDataset(Dataset):
    def __init__(self, ann_dir, img_dir, max_samples=None):
        self.ann_dir  = ann_dir
        self.img_dir  = img_dir
        ann_files     = sorted([f for f in os.listdir(ann_dir) if f.endswith('.json')])

        self.valid_ids = []
        for fname in tqdm(ann_files[:max_samples] if max_samples else ann_files,
                          desc="Scanning annotations"):
            img_id   = fname.replace('.json', '')
            img_path = os.path.join(img_dir, img_id + '.jpg')
            if not os.path.exists(img_path):
                continue
            with open(os.path.join(ann_dir, fname)) as f:
                ann = json.load(f)
            has_top5 = any(
                item.get('category_id') in TOP5_CATEGORIES
                for key, item in ann.items()
                if key.startswith('item')
            )
            if has_top5:
                self.valid_ids.append(img_id)
        print(f"Found {len(self.valid_ids)} valid images")

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):
        img_id   = self.valid_ids[idx]
        img_path = os.path.join(self.img_dir, img_id + '.jpg')
        ann_path = os.path.join(self.ann_dir, img_id + '.json')

        img            = Image.open(img_path).convert('RGB')
        orig_w, orig_h = img.size
        img            = img.resize((512, 512))
        img_w, img_h   = 512, 512
        img_tensor     = F.to_tensor(img)
        scale_x        = img_w / orig_w
        scale_y        = img_h / orig_h

        with open(ann_path) as f:
            ann = json.load(f)

        boxes, labels, masks = [], [], []
        for key, item in ann.items():
            if not key.startswith('item'):
                continue
            cat_id = item.get('category_id')
            if cat_id not in TOP5_CATEGORIES:
                continue
            bbox = item.get('bounding_box', [])
            segs = item.get('segmentation', [])
            if not bbox or not segs:
                continue
            x1, y1, x2, y2 = bbox
            x1 = max(0, min(x1 * scale_x, img_w - 1))
            y1 = max(0, min(y1 * scale_y, img_h - 1))
            x2 = max(0, min(x2 * scale_x, img_w - 1))
            y2 = max(0, min(y2 * scale_y, img_h - 1))
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append([x1, y1, x2, y2])
            labels.append(TOP5_CATEGORIES[cat_id])
            mask = np.zeros((img_h, img_w), dtype=np.uint8)
            for polygon in segs:
                if len(polygon) < 6:
                    continue
                px = np.clip(np.array(polygon[0::2], dtype=np.float32) * scale_x, 0, img_w - 1)
                py = np.clip(np.array(polygon[1::2], dtype=np.float32) * scale_y, 0, img_h - 1)
                rr, cc = skimage.draw.polygon(py, px, shape=(img_h, img_w))
                mask[rr, cc] = 1
            masks.append(mask)

        if len(boxes) == 0:
            target = {
                'boxes'   : torch.zeros((0, 4), dtype=torch.float32),
                'labels'  : torch.zeros(0, dtype=torch.int64),
                'masks'   : torch.zeros((0, img_h, img_w), dtype=torch.uint8),
                'image_id': torch.tensor([idx])
            }
        else:
            target = {
                'boxes'   : torch.tensor(boxes, dtype=torch.float32),
                'labels'  : torch.tensor(labels, dtype=torch.int64),
                'masks'   : torch.tensor(np.array(masks), dtype=torch.uint8),
                'image_id': torch.tensor([idx])
            }
        return img_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


print("Building train dataset...")
train_dataset = ApparelMaskRCNNDataset(TRAIN_ANN, TRAIN_IMG, max_samples=5000)
print("Building val dataset...")
val_dataset   = ApparelMaskRCNNDataset(VAL_ANN, VAL_IMG)

# pin_memory + increased workers for faster data loading
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          num_workers=4, collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False,
                          num_workers=4, collate_fn=collate_fn, pin_memory=True)

print(f"\nTrain: {len(train_dataset)} | Val: {len(val_dataset)}")

Building train dataset...


Scanning annotations: 100%|██████████| 5000/5000 [01:04<00:00, 78.00it/s]


Found 3678 valid images
Building val dataset...


Scanning annotations: 100%|██████████| 32153/32153 [07:07<00:00, 75.21it/s]

Found 23741 valid images

Train: 3678 | Val: 23741


In [3]:
# Build Model
def get_state_dict(model):
    return model.module.state_dict() if hasattr(model, 'module') else model.state_dict()

def build_maskrcnn():
    model = maskrcnn_resnet50_fpn(weights=None)
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, NUM_CLASSES)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, NUM_CLASSES)
    return model

model = build_maskrcnn().to(DEVICE)

print("Mask R-CNN loaded FROM SCRATCH on single GPU")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print("All layers trainable")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]


Mask R-CNN loaded FROM SCRATCH on single GPU
Parameters: 43.94M
All layers trainable


In [4]:
# Training Setup

NUM_EPOCHS = 12

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.01, momentum=0.9, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)
scaler    = GradScaler()  # for AMP

print(f"Optimizer : SGD | LR: 0.01 | Epochs: {NUM_EPOCHS}")
print("AMP GradScaler ready")

Optimizer : SGD | LR: 0.01 | Epochs: 12
AMP GradScaler ready


In [5]:
# Training Loop

from tqdm import tqdm

START_EPOCH = 1
best_loss   = float('inf')
resume_path = SAVE_DIR + 'maskrcnn_scratch_latest.pt'

if os.path.exists(resume_path):
    print("Found checkpoint — resuming scratch training...")
    checkpoint = torch.load(resume_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    START_EPOCH = checkpoint['epoch'] + 1
    best_loss   = checkpoint['best_loss']
    print(f"Resumed from epoch {checkpoint['epoch']} | Best loss: {best_loss:.4f}")
else:
    print("No checkpoint — starting fresh scratch training")

for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    train_bar  = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [Train]")

    for imgs, targets in train_bar:
        imgs    = [img.to(DEVICE) for img in imgs]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        with autocast():
            loss_dict = model(imgs, targets)
            losses    = sum(loss for loss in loss_dict.values())

        scaler.scale(losses).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += losses.item()
        train_bar.set_postfix(loss=f"{losses.item():.4f}")

    epoch_loss /= len(train_loader)
    scheduler.step()
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Loss: {epoch_loss:.4f}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(get_state_dict(model), SAVE_DIR + 'maskrcnn_scratch_best.pt')
        print(f"  ✓ Best model saved (Loss: {best_loss:.4f})")

    torch.save({
        'epoch'               : epoch,
        'model_state_dict'    : get_state_dict(model),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict'   : scaler.state_dict(),
        'best_loss'           : best_loss,
    }, SAVE_DIR + 'maskrcnn_scratch_latest.pt')
    print(f"  ✓ Checkpoint saved (epoch {epoch})")

    torch.cuda.empty_cache()  # prevent memory buildup across epochs

print("\nScratch training complete!")

No checkpoint — starting fresh scratch training


Epoch 01/12 [Train]: 100%|██████████| 460/460 [06:29<00:00,  1.18it/s, loss=3.5316]


Epoch 01/12 | Loss: 3.6361
  ✓ Best model saved (Loss: 3.6361)
  ✓ Checkpoint saved (epoch 1)


Epoch 02/12 [Train]: 100%|██████████| 460/460 [06:32<00:00,  1.17it/s, loss=3.5264]


Epoch 02/12 | Loss: 3.5984
  ✓ Best model saved (Loss: 3.5984)
  ✓ Checkpoint saved (epoch 2)


...

Epoch 10/12 | Loss: 3.5021
  ✓ Best model saved (Loss: 3.5021)
  ✓ Checkpoint saved (epoch 10)


Epoch 11/12 [Train]: 100%|██████████| 460/460 [06:32<00:00,  1.17it/s, loss=3.4693]


Epoch 11/12 | Loss: 3.5028
  ✓ Checkpoint saved (epoch 11)


Epoch 12/12 [Train]: 100%|██████████| 460/460 [06:32<00:00,  1.17it/s, loss=3.5050]


Epoch 12/12 | Loss: 3.5031
  ✓ Checkpoint saved (epoch 12)

Scratch training complete!


In [6]:
# Evaluation metrics

from torchvision.ops import box_iou

def evaluate_maskrcnn(model, loader, iou_threshold=0.5):
    model.eval()
    all_ious, all_dice = [], []
    class_tp = {i: 0 for i in range(1, 6)}
    class_fp = {i: 0 for i in range(1, 6)}
    class_fn = {i: 0 for i in range(1, 6)}

    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc='Evaluating'):
            imgs  = [img.to(DEVICE) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes  = target['boxes'].to(DEVICE)
                gt_labels = target['labels'].to(DEVICE)
                gt_masks  = target['masks'].to(DEVICE).float()

                if len(pred['boxes']) == 0 or len(gt_boxes) == 0:
                    continue

                pred_boxes  = pred['boxes']
                pred_labels = pred['labels']
                pred_masks  = (pred['masks'][:, 0] > 0.5).float()
                pred_scores = pred['scores']

                keep        = pred_scores > 0.5
                pred_boxes  = pred_boxes[keep]
                pred_labels = pred_labels[keep]
                pred_masks  = pred_masks[keep]

                if len(pred_boxes) == 0:
                    continue

                iou_matrix = box_iou(pred_boxes, gt_boxes)
                matched_gt = set()

                for p_idx in range(len(pred_boxes)):
                    best_iou, best_gt = iou_matrix[p_idx].max(0)
                    best_gt  = best_gt.item()
                    best_iou = best_iou.item()
                    p_label  = pred_labels[p_idx].item()
                    g_label  = gt_labels[best_gt].item() if best_gt < len(gt_labels) else -1

                    if best_iou >= iou_threshold and p_label == g_label and best_gt not in matched_gt:
                        matched_gt.add(best_gt)
                        class_tp[p_label] = class_tp.get(p_label, 0) + 1
                        if best_gt < len(gt_masks) and p_idx < len(pred_masks):
                            pm = pred_masks[p_idx]
                            gm = gt_masks[best_gt]
                            if pm.shape == gm.shape:
                                intersection = (pm * gm).sum().item()
                                union        = (pm + gm).clamp(0, 1).sum().item()
                                all_ious.append(intersection / (union + 1e-6))
                                all_dice.append(2 * intersection / (pm.sum().item() + gm.sum().item() + 1e-6))
                    else:
                        class_fp[p_label] = class_fp.get(p_label, 0) + 1

                for g_idx in range(len(gt_boxes)):
                    if g_idx not in matched_gt:
                        g_label = gt_labels[g_idx].item()
                        class_fn[g_label] = class_fn.get(g_label, 0) + 1

    print("\n── Per-class Detection Results ──")
    print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("-" * 45)
    macro_f1s = []
    for cls_idx in range(1, 6):
        tp = class_tp[cls_idx]
        fp = class_fp[cls_idx]
        fn = class_fn[cls_idx]
        p  = tp / (tp + fp + 1e-6)
        r  = tp / (tp + fn + 1e-6)
        f1 = 2 * p * r / (p + r + 1e-6)
        macro_f1s.append(f1)
        print(f"{IDX_TO_NAME[cls_idx]:<20} {p:>10.4f} {r:>10.4f} {f1:>10.4f}")

    print(f"\nMacro F1      : {np.mean(macro_f1s):.4f}")
    print(f"Mean mask IoU : {np.mean(all_ious):.4f}")
    print(f"Mean Dice     : {np.mean(all_dice):.4f}")
    return np.mean(macro_f1s), np.mean(all_ious), np.mean(all_dice)

# Load best scratch model
model.load_state_dict(torch.load(SAVE_DIR + 'maskrcnn_scratch_best.pt', map_location=DEVICE))
macro_f1, miou, dice = evaluate_maskrcnn(model, val_loader)

Evaluating: 100%|██████████| 2968/2968 [1:08:44<00:00,  1.39s/it]


── Per-class Detection Results ──
Class                 Precision     Recall         F1
---------------------------------------------
short_sleeve_top         0.0000     0.0000     0.0000
trousers                 0.0000     0.0000     0.0000
shorts                   0.0000     0.0000     0.0000
long_sleeve_top          0.0000     0.0000     0.0000
skirt                    0.0000     0.0000     0.0000

Macro F1      : 0.0000
Mean mask IoU : nan
Mean Dice     : nan


In [7]:
# Save and upload

import subprocess

os.makedirs('/kaggle/working/maskrcnn_scratch_upload/', exist_ok=True)

shutil.copy(SAVE_DIR + 'maskrcnn_scratch_best.pt',
            '/kaggle/working/maskrcnn_scratch_upload/maskrcnn_scratch_best.pt')

metrics = {
    'macro_f1': float(macro_f1),
    'mIoU'    : float(miou),
    'dice'    : float(dice),
    'note'    : 'Mask R-CNN from scratch, 5000 samples, 12 epochs, DataParallel + AMP'
}
with open('/kaggle/working/maskrcnn_scratch_upload/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

metadata = {
    "title"   : "vr-maskrcnn-scratch-model",
    "id"      : "pankajdeopa/vr-maskrcnn-scratch-model",
    "licenses": [{"name": "CC0-1.0"}]
}
with open('/kaggle/working/maskrcnn_scratch_upload/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', '/kaggle/working/maskrcnn_scratch_upload/',
     '--dir-mode', 'zip'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

Starting upload for file metrics.json
Upload successful: metrics.json (133B)
Starting upload for file maskrcnn_scratch_best.pt
Upload successful: maskrcnn_scratch_best.pt (168MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopa/vr-maskrcnn-scratch-model


  0%|          | 0.00/133 [00:00<?, ?B/s]
100%|██████████| 133/133 [00:00<00:00, 378B/s]

  0%|          | 0.00/168M [00:00<?, ?B/s]
  5%|▍         | 8.17M/168M [00:00<00:02, 78.9MB/s]
 18%|█▊        | 29.8M/168M [00:00<00:01, 128MB/s] 
 29%|██▊       | 48.0M/168M [00:00<00:00, 137MB/s]
 36%|███▌      | 60.8M/168M [00:00<00:00, 135MB/s]
 44%|████▎     | 73.5M/168M [00:00<00:00, 113MB/s]
 52%|█████▏    | 88.2M/168M [00:00<00:00, 122MB/s]
 60%|█████▉    | 100M/168M [00:00<00:00, 114MB/s] 
 67%|██████▋   | 112M/168M [00:00<00:00, 116MB/s]
 75%|███████▌  | 126M/168M [00:01<00:00, 122MB/s]
 84%|████████▍ | 141M/168M [00:01<00:00, 131MB/s]
 92%|█████████▏| 154M/168M [00:01<00:00, 12